In [ ]:
# The following contents are for testing if the model's forward pass works well

In [1]:
import torch
import h5py
import numpy as np
from NN_model import BehaviorPredictor   # uses the version now in your repo

# ---------------------------------------------------------------------
# Quick sanity test: forward pass on a single video entry
# ---------------------------------------------------------------------

h5_path = r"D:\Mouse_behavior_data\D21\input_output_data_downsample_444.h5"

# Read one sample (e.g., first video)
with h5py.File(h5_path, "r") as f:
    first_key = list(f.keys())[0]
    g = f[first_key]
    X = g["X"][:]            # (3, H, W, T)
    y = g["y"][:]            # (num_labels, T)
    stim_type = g["stim_type"][()]
    stim_loc = g["stim_location"][()]
    stim_onset = g["stim_onset"][()]
    stim_offset = g["stim_offset"][()]
    stim_sentence = g["stim_sentence"][()] if "stim_sentence" in g else b""
    stim_sentence = stim_sentence.decode() if isinstance(stim_sentence, (bytes, np.bytes_)) else str(stim_sentence)

print("Loaded:", first_key, "X shape:", X.shape, "y shape:", y.shape)
print("Stim sentence:", stim_sentence)

Loaded: video_01 X shape: (3, 207, 133, 138) y shape: (9, 138)
Stim sentence: [b'Stimulation: brush on ipsilateral hindpaw.']


In [2]:
# Convert to torch tensor in the correct shape (C, T, H, W)
X = np.transpose(X, (0, 3, 1, 2))   # (3, T, H, W)
X = torch.tensor(X, dtype=torch.float32).unsqueeze(0)   # (B=1, 3, T, H, W)

# 1) Ensure y is (T, num_labels)
if y.shape[0] < y.shape[1]:   # (9, 138) → transpose
    y = y.T                   # (138, 9)
y = torch.tensor(y, dtype=torch.float32).unsqueeze(0)   # (B=1, T, num_labels)

stim_type   = torch.tensor([stim_type], dtype=torch.long)
stim_loc    = torch.tensor([stim_loc], dtype=torch.long)
stim_onset  = torch.tensor([stim_onset], dtype=torch.int32)
stim_offset = torch.tensor([stim_offset], dtype=torch.int32)
lengths     = torch.tensor([X.shape[2]], dtype=torch.int32)  # full length

# ---------------------------------------------------------------------
# Instantiate model
# ---------------------------------------------------------------------
num_labels = y.shape[-1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BehaviorPredictor(
    num_labels=num_labels,
    use_text_encoder=True,
    text_mode="sentence_transformer",
    text_model_name="sentence-transformers/all-MiniLM-L6-v2"
).to(device)

# ---------------------------------------------------------------------
# Forward pass
# ---------------------------------------------------------------------
model.eval()
with torch.no_grad():
    logits = model(
        X.to(device),
        stim_type.to(device),
        stim_loc.to(device),
        stim_onset.to(device),
        stim_offset.to(device),
        lengths.to(device),
        sent_text=[stim_sentence]
    )

print("Forward pass OK → logits shape:", logits.shape)

C:\Users\james\AppData\Local\Temp\ipykernel_21148\1235489334.py:10: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  stim_type   = torch.tensor([stim_type], dtype=torch.long)


Forward pass OK → logits shape: torch.Size([1, 138, 9])


In [3]:
logits.shape

torch.Size([1, 138, 9])

In [4]:
y.shape

torch.Size([1, 138, 9])